**Introduction**

_The purpose of this analysis is to identify which guest segments generate the most ancillary revenue (spa, dining, and activities) at Solstice Opal Hotel, and to provide actionable recommendations that leadership can use to maximise this revenue stream. The analysis draws on three datasets: guest profiles, stay details, and ancillary spend records._

In [ ]:
#import libraries
import pandas as pd          
import numpy as np          
import matplotlib
matplotlib.use('Agg')        
import matplotlib.pyplot as plt  
import matplotlib.ticker as mticker  
import re                    
import warnings
warnings.filterwarnings('ignore')


**Download and Explore the Data**

In [ ]:
#Load the data
stay_details = pd.read_csv('da_sample_stay_details.csv')
guest_profiles = pd.read_csv('da_sample_guest_profiles.csv')
spend = pd.read_csv('da_sample_ancillary_spend.csv')

In [ ]:
# Rows and columns for each dataset
print("Data loaded")
print(f"   guest_profiles : {guest_profiles.shape[0]} rows, {guest_profiles.shape[1]} columns")
print(f"   stay_details   : {stay_details.shape[0]} rows, {stay_details.shape[1]} columns")
print(f"   ancillary_spend: {spend.shape[0]} rows, {spend.shape[1]} columns")

In [ ]:
#Guest_profile Check

print("\n" + "="*60)
print("GUEST PROFILES")
print("="*60)
print(guest_profiles.head())
print("\nData types:")
print(guest_profiles.dtypes)
print("\nMissing values per column:")
print(guest_profiles.isnull().sum())

In [ ]:
#Stay Details Check

print('\n' + '=' * 60)
print('STAY DETAILS')
print('=' * 60)
print(stay_details.head())
print('\nData Types:')
print(stay_details.dtypes)
print('\nMissing Values per column:')
print(stay_details.isnull().sum())

In [ ]:
#Ancillary Spend

print('\n' + '=' * 60)
print('ANCILLARY SPEND')
print('=' * 60)
print(spend.head())
print('\Data Types:')
print(spend.dtypes)
print('\nMissing Value per Column:')
print(spend.isnull().sum())

**DATA VALIDATION AND CLEANING**

In [ ]:
#---Guest Profile---
print("\n" + "="*60)
print("VALIDATION: GUEST PROFILES")
print("="*60)
print(f"\nguest_id duplicates : {guest_profiles['guest_id'].duplicated().sum()}")
print(f"guest_id nulls      : {guest_profiles['guest_id'].isnull().sum()}")
print(f"guest_id range      : {guest_profiles['guest_id'].min()} to {guest_profiles['guest_id'].max()}")
print(f"\nloyalty_tier unique values:\n{guest_profiles['loyalty_tier'].value_counts(dropna=False)}")
print(f"\nmarketing_consent values:\n{guest_profiles['marketing_consent'].value_counts(dropna=False)}")

In [ ]:
#---Stay Details---

print("\n" + "="*60)
print("VALIDATION: STAY DETAILS")
print("="*60)

#guest_id in stay_details vs guest_id in guest_profile
guests_only_in_stays = set(stay_details['guest_id']) - set(guest_profiles['guest_id'])
print(f"\nGuest IDs in stays but NOT in profiles: {len(guests_only_in_stays)}")

# stay_id duplicates
print(f"stay_id duplicates: {stay_details['stay_id'].duplicated().sum()}")
 
#booking_channel
print(f"\nbooking_channel values:\n{stay_details['booking_channel'].value_counts(dropna=False)}")
 
#reason_for_stay
print(f"\nreason_for_stay values:\n{stay_details['reason_for_stay'].value_counts(dropna=False)}")

# --- number_of_guests ---
# We check for zero or negative values — a stay with 0 guests makes no sense.
print(f"\nnumber_of_guests value counts:\n{stay_details['number_of_guests'].value_counts().sort_index()}")
print(f"Zero or negative number_of_guests: {(stay_details['number_of_guests'] <= 0).sum()}")

# --- check for overlapping stays ---
# We convert check_in_date to a proper datetime format first.
stay_details['check_in_date'] = pd.to_datetime(stay_details['check_in_date'])
stays_sorted = stay_details.sort_values(['guest_id', 'check_in_date'])
 
#compare each stay's date to the PREVIOUS stay's date.
stays_sorted['prev_checkin'] = stays_sorted.groupby('guest_id')['check_in_date'].shift(1)
 
# Calculate the gap in days between consecutive stays for the same guest
stays_sorted['days_gap'] = (stays_sorted['check_in_date'] - stays_sorted['prev_checkin']).dt.days

# Flag exact same-day check-ins 
overlapping = stays_sorted[stays_sorted['days_gap'] == 0]
print(f"\nOverlapping stays (same check-in date, same guest): {len(overlapping)}")
 
# Flag stays within 1 day of each other
close_stays = stays_sorted[(stays_sorted['days_gap'] >= 0) & (stays_sorted['days_gap'] <= 1)]
print(f"Stays within 1 day of each other (same guest): {len(close_stays)}")



In [ ]:
print("\n" + "="*60)
print("VALIDATION: ANCILLARY SPEND")
print("="*60)
 
print(f"\nActual column names in spend: {list(spend.columns)}")
 
# --- guest_id values have letter suffixes ---
print(f"\nSample guest_id values: {spend['guest_id'].head(10).tolist()}")

# str[-1] extracts the last character of every string in the column
print(f"\nLast character (suffix) counts:")
print(spend['guest_id'].str[-1].value_counts())

# Extract just the numeric part using regex
spend['guest_id_numeric'] = spend['guest_id'].str.extract(r'(\d+)').astype(float)
print(f"\nAfter stripping suffix — ID range: {spend['guest_id_numeric'].min()} to {spend['guest_id_numeric'].max()}")
 
# Check if any of these IDs don't exist in the guest profiles
unmatched = set(spend['guest_id_numeric'].dropna().astype(int)) - set(guest_profiles['guest_id'])
print(f"IDs in spend not found in profiles (after cleaning): {len(unmatched)}")
 
# --- category ---
print(f"\ncategory values:\n{spend['category'].value_counts(dropna=False)}")
 
# --- amount_spent ---
print(f"\namount_spent nulls    : {spend['amount_spent'].isnull().sum()}")
print(f"amount_spent negatives: {(spend['amount_spent'] < 0).sum()}")
print(f"amount_spent zeros    : {(spend['amount_spent'] == 0).sum()}")
print(f"\namount_spent statistics:\n{spend['amount_spent'].describe()}")

**DATA CLEANING**

In [ ]:
print("\n" + "="*60)
print("DATA CLEANING")
print("="*60)

# --- Clean 1: Fix ancillary_spend guest_id ---
spend['guest_id'] = spend['guest_id'].astype(str).str.extract(r'(\d+)').astype(int)
print(f"\n Stripped letter suffixes from ancillary_spend guest_id")
print(f"   Sample after fix: {spend['guest_id'].head(5).tolist()}")

# --- Clean 2: Drop rows with null amount_spent ---
rows_before = len(spend)
spend_clean = spend.dropna(subset=['amount_spent']).copy()
rows_after = len(spend_clean)
print(f"\n Dropped {rows_before - rows_after} rows with null amount_spent")
print(f"   Spend rows: {rows_before} → {rows_after}")

# --- Clean 3: Remove duplicate stay_ids ---
rows_before = len(stay_details)
stays_clean = stay_details.drop_duplicates(subset=['stay_id'], keep='first').copy()
rows_after = len(stays_clean)
print(f"\n Removed {rows_before - rows_after} duplicate stay_id rows")
print(f"   Stays rows: {rows_before} → {rows_after}")

# Make sure check_in_date is datetime in the clean version too
stays_clean['check_in_date'] = pd.to_datetime(stays_clean['check_in_date'])

# --- Clean 4: Fill null loyalty_tier with 'Non-member' ---
guests_clean = guest_profiles.copy()
nulls_before = guests_clean['loyalty_tier'].isnull().sum()
guests_clean['loyalty_tier'] = guests_clean['loyalty_tier'].fillna('Non-member')
print(f"\n Filled {nulls_before} null loyalty_tier values with 'Non-member'")
print(f"   Loyalty tier distribution after fix:")
print(f"   {guests_clean['loyalty_tier'].value_counts().to_dict()}")

**MERGE THE THREE TABLES**

In [ ]:
print("\n" + "="*60)
print("MERGING TABLES")
print("="*60)
 
# --- Merge 1: Stays + Guest Profiles ---
merged = stays_clean.merge(guests_clean, on='guest_id', how='left')
print(f"\nMerged stays + guests: {merged.shape[0]} rows")
 
# --- Aggregate spend per guest ---
spend_agg = spend_clean.groupby('guest_id').agg(
    total_ancillary_spend = ('amount_spent', 'sum'),    # sum all amounts
    num_transactions      = ('amount_spent', 'count')  # count how many transactions
).reset_index() 
 
print(f" Aggregated spend per guest: {spend_agg.shape[0]} guests with spend records")
print(f"   Sample:\n{spend_agg.head()}")
 
# --- Merge 2: Add spend to the main table ---
full = merged.merge(spend_agg, on='guest_id', how='left')
 
# Guests with NO spend records will have NaN — replace with 0
# These are the "zero-spend" guests — they stayed but bought nothing.
full['total_ancillary_spend'] = full['total_ancillary_spend'].fillna(0)
full['num_transactions']      = full['num_transactions'].fillna(0)
 
print(f" Final merged table: {full.shape[0]} rows x {full.shape[1]} columns")
print(f"\nFinal table columns: {list(full.columns)}")
print(f"\nSample of final table:")
print(full[['guest_id','loyalty_tier','reason_for_stay','booking_channel','total_ancillary_spend']].head(8))

**EXPLORATORY DATA ANALYSIS**

In [ ]:
print("\n" + "="*60)
print("EXPLORATORY ANALYSIS — KEY NUMBERS")
print("="*60)
 
# --- Headline numbers ---
total_revenue   = full['total_ancillary_spend'].sum()
total_stays     = len(full)
avg_per_stay    = total_revenue / total_stays
zero_spend      = (full['total_ancillary_spend'] == 0).sum()
zero_spend_pct  = 100 * zero_spend / total_stays
 
print(f"\nHEADLINE NUMBERS")
print(f"   Total ancillary revenue : ${total_revenue:,.2f}")
print(f"   Total stays             : {total_stays}")
print(f"   Avg revenue per stay    : ${avg_per_stay:.2f}  -- THIS IS OUR KEY METRIC (AARS)")
print(f"   Zero-spend stays        : {zero_spend} ({zero_spend_pct:.1f}%)")
 
# --- By loyalty tier ---
tier_order = ['Non-member', 'Silver', 'Gold', 'Platinum']
 
tier_analysis = full.groupby('loyalty_tier')['total_ancillary_spend'].agg(
    avg_spend  = 'mean',
    total_spend= 'sum',
    num_stays  = 'count'
).reindex(tier_order)  # .reindex() forces the rows to appear in our chosen order
 
print(f"\nBY LOYALTY TIER")
print(tier_analysis.round(2))
 
# --- By reason for stay ---
reason_analysis = full.groupby('reason_for_stay')['total_ancillary_spend'].agg(
    avg_spend  = 'mean',
    total_spend= 'sum',
    num_stays  = 'count'
)
print(f"\nBY REASON FOR STAY")
print(reason_analysis.round(2))
 
# --- By booking channel ---
channel_analysis = full.groupby('booking_channel')['total_ancillary_spend'].agg(
    avg_spend  = 'mean',
    total_spend= 'sum',
    num_stays  = 'count'
).sort_values('avg_spend', ascending=False)
print(f"\nBY BOOKING CHANNEL")
print(channel_analysis.round(2))
 
# --- By category (from spend_clean, not full) ---
# We use spend_clean here because we want per-transaction category data
cat_spend = spend_clean.groupby('category')['amount_spent'].sum().sort_values(ascending=False)
print(f"\nTOTAL REVENUE BY CATEGORY")
print(cat_spend.round(2))
 
# --- Loyalty tier x reason for stay (multivariate) ---
pivot = full.groupby(['loyalty_tier', 'reason_for_stay'])['total_ancillary_spend'].mean().unstack()
pivot = pivot.reindex(tier_order)
print(f"\n AVG SPEND BY TIER AND REASON FOR STAY")
print(pivot.round(2))
 